In [158]:
import pickle
from FlyOutput import FlyOutput
import Plotters
import plotly.graph_objects as go
import numpy as np  
import Utils
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import time
import pandas as pd
from PoseEstimation import PoseEstimation
import matplotlib.cm as cm

%matplotlib qt





path_output = 'I:/My Drive/Research/gaussian_splatting/gaussian_splatting_output/'

model_name = 'fly_model_to_fly'
file_name = 'fly_model'

dict_path  = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/frames_model.pkl'
image_path = 'G:/My Drive/Research/gaussian_splatting/gaussian_splatting_input/mov30_2024_11_12_darkan/'



path_angles = f'{path_output}/{model_name}/{file_name}_angles.pkl'
path_results = f'{path_output}/{model_name}/{file_name}_angles.pkl'


# download model_run localy
output_angles_weights_path = 'D:/Documents/gaussian_model_output/fly_model_to_fly/fly_model_results.pkl'

if os.path.exists(f'{output_angles_weights_path}'):
    with open(output_angles_weights_path, 'rb') as handle:
        output_angles_weights = pickle.load(handle)

iteration = 1200

frame0 = 1430
frame_end = 1447
frame = 1450
weight_flag = False

with open(dict_path,'rb') as f:
    frames = pickle.load(f)





input_dir = 'D:/Documents/gaussian_model_output/fly_model_to_fly'


model_name = 'fly_model_to_fly'


angle_name = ['phi','theta','psi','phi','psi','yaw','pitch']
letedict = {'num_of_bins' : 20,'perc_wing_for_le' : 1, 'wing_length_snip':0.27}




with open(dict_path,'rb') as f:
    frames = pickle.load(f)
iterations = 1000


frame_output = FlyOutput(image_path,frame,input_dir,output_angles_weights,frame0,iteration,f'fly_model',letedict = letedict,deg = 0,skip_frames = 1,frames_dict = frames)



In [226]:
approx_span, approx_chord = frame_output.wing_span_chord(frame_output.right_wing)

mean_wing = np.mean(frame_output.right_wing ,axis = 0)
projected_span = np.dot(frame_output.right_wing - mean_wing,approx_span)
tip_side = np.max(projected_span)
root_side = np.min(projected_span)
wing_size = np.abs(tip_side - root_side)
tip_chunck = frame_output.right_wing[projected_span > (tip_side - wing_size*0.1),:]
tip_mean = np.mean(tip_chunck,axis = 0)
span = (tip_mean - mean_wing ) / np.linalg.norm(tip_mean - mean_wing)

In [227]:
projected_chord = np.dot(frame_output.right_wing - mean_wing,approx_chord)
le = frame_output.right_wing[projected_chord > 0]
te = frame_output.right_wing[projected_chord <= 0]

In [228]:
span_to_plot = np.vstack((mean_wing,mean_wing+span/1000) )
chord_to_plot = np.vstack((mean_wing,mean_wing+approx_chord/1000) )

le_projected_span = np.dot(le - mean_wing,approx_span)
te_projected_span = np.dot(te - mean_wing,approx_span)

le_projected_chord = np.dot(le - mean_wing,approx_chord)
te_projected_chord = np.dot(te - mean_wing,approx_chord)


num_of_bins =50

def bin_le_te(projected_le_te):
    diff = (max(projected_le_te) - min(projected_le_te))/num_of_bins
    bin_edges = np.arange(np.min(projected_le_te), np.max(projected_le_te) + diff, diff)
    return np.digitize(projected_le_te, bins=bin_edges)

def get_max_le_te(bin_indices_le,idx,projected_chord,le_te):
    le_bin = le_te[bin_indices_le == idx]
    if len(le_bin) > 0:
        return le_bin[np.argmax(projected_chord[bin_indices_le == idx])]
    else:
        return [None]*3


bin_indices_le = bin_le_te(le_projected_span)
bin_indices_te = bin_le_te(te_projected_span)

le_bins = [get_max_le_te(bin_indices_le,idx,le_projected_chord,le) for idx in bin_indices_le] 
te_bins = [get_max_le_te(bin_indices_te,idx,-te_projected_chord,te) for idx in bin_indices_te] 

le_bins = np.vstack(le_bins)
te_bins = np.vstack(te_bins)

In [256]:
from skimage.measure import LineModelND, ransac,EllipseModel


le_projected_span = np.dot(le_bins - mean_wing,span)
# tip_side = np.max(le_projected_span)
root_side = np.min(le_projected_span)



root_le = le_bins[le_projected_span < (root_side + wing_size*0.5)]


model_robust, inliers = ransac(root_le, LineModelND, min_samples=2, residual_threshold=5/100000, max_trials=1000)
origin, direction = model_robust.params

le_ransac = np.vstack((origin-direction/1000,origin+direction/1000))

In [254]:
le_ransac
fig = go.Figure()
Plotters.scatter3d(fig,frame_output.body,'green',4,'body',show_colorbar = False, opa=1)
# Plotters.scatter3d(fig,frame_output.right_wing,'red',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,frame_output.left_wing,'blue',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,frame_output.left_wing_te,'black',4,'body',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,le,'pink',1,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,te,'purple',1,'body',show_colorbar = False, opa=1)

Plotters.scatter3d(fig,te_bins,'orange',4,'body',show_colorbar = False, opa=1)


Plotters.scatter3d(fig,root_le,'magenta',4,'body',show_colorbar = False, opa=1)
Plotters.scatter3d(fig,root_le[inliers],'red',4,'body',show_colorbar = False, opa=1)


Plotters.scatter3d(fig,np.atleast_2d(tip_mean),'green',10,'span',show_colorbar = False, opa=1, mode = 'markers')
Plotters.scatter3d(fig,span_to_plot,'black',20,'span',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,chord_to_plot,'black',20,'span',show_colorbar = False, opa=1, mode = 'lines')
Plotters.scatter3d(fig,le_ransac,'black',20,'span',show_colorbar = False, opa=1, mode = 'lines')

fig.data[-1].line.width = 15
fig.data[-2].line.width = 15

fig.show()

In [91]:
tip_side - tip_side*0.1

0.0010308775430234456

In [28]:
frame_output.left_wing_le

array([[ 0.01529814, -0.00500524, -0.00517526],
       [ 0.01463135, -0.00504553, -0.00526702],
       [ 0.0149807 , -0.00499444, -0.00519358],
       [ 0.01475105, -0.0048789 , -0.00527208],
       [ 0.01516035, -0.00498715, -0.00519646],
       [ 0.01443851, -0.00508401, -0.00531252],
       [ 0.01476897, -0.00502234, -0.0052291 ],
       [ 0.01539292, -0.00501493, -0.0051688 ],
       [ 0.01504768, -0.00497888, -0.00519747],
       [ 0.01437093, -0.00508556, -0.00536016],
       [ 0.01562783, -0.00509406, -0.00520695],
       [ 0.01555548, -0.00505666, -0.0051898 ],
       [ 0.0141794 , -0.00513633, -0.00532361],
       [ 0.0157557 , -0.00508911, -0.00526242],
       [ 0.01417457, -0.00513234, -0.00536819],
       [ 0.01588038, -0.00532987, -0.00536284],
       [ 0.01550299, -0.00560715, -0.00567146],
       [ 0.01537342, -0.00565178, -0.00591145],
       [ 0.01559832, -0.00556043, -0.00576446],
       [ 0.01520502, -0.00564913, -0.00582946],
       [ 0.01574022, -0.00546803, -0.005